# 10  Faisons-nous aussi bien que le PMQ ? Analyse avec les arbitrages du SeRN

**Notebook autonome.** Il charge lui-même le Fiqual, le fichier des divergences et les images
depuis S3. Pour RG25, il **réutilise les lectures de gemma4 du notebook 09** (aucun nouvel appel) ;
.

## Le problème de toutes les mesures précédentes

Jusqu'ici, la  vérité  était le **Fiqual**, c'est-à-dire **la saisie du PMQ elle-même**. Or le
PMQ se trompe aussi. Deux conséquences :

- quand gemma4 lit **comme le PMQ** et que le PMQ s'est trompé, gemma4 était compté **juste** à tort ;
- quand gemma4 lit **mieux que le PMQ**, il était compté **faux** à tort.

Le SeRN compare les deux saisies (PAD → Ficod, PMQ → Fiqual) et **arbitre** leurs désaccords. On peut
donc construire une **vérité arbitrée**, plus juste que le Fiqual seul.



In [26]:
# 1. Reglages et outils (tout est ici)
import base64
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import zipfile
from pathlib import Path

import mlflow
import pandas as pd
import requests
from PIL import Image
from scipy import stats

CAMPAGNE = None                  # None = RG25 si elle a un fichier de divergences, sinon RG26
CAMPAGNES_POSSIBLES = ["RG25", "RG26"]
TEST_RAPIDE = False              # True : 10 bulletins (verification)
MODELE = "gemma4-26b-moe"
VARIABLE_NOM, VARIABLE_PRENOM = None, None     # None = detection automatique (colonne VARIABLE)
LONGUEUR_MIN_CORRECTION = 4

BUCKET = "projet-mesure-qualite-rp"
POS_UT, POS_CAB = 2, 4
COLS = {"NOM": 11, "PRENOM": 12}
BORDS_BLOC = (0.075, 0.072, 0.450, 0.105)
g, h, d_, b_ = BORDS_BLOC
ZONE_BLOC = (g, h, d_ - g, b_ - h)

RESULTATS = Path.home() / "work" / "banc-htr" / "resultats"
LISTES = Path.home() / "work" / "banc-htr" / "listes_insee"
BLOCS = Path("/tmp/arbitrage/blocs")
for d in [RESULTATS, LISTES, BLOCS]:
    d.mkdir(parents=True, exist_ok=True)
RACINE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
SUFFIXE = "_rapide" if TEST_RAPIDE else ""


def normaliser(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    t = unicodedata.normalize("NFKD", str(t).strip())
    t = "".join(c for c in t if not unicodedata.combining(c)).upper()
    t = re.sub(r"[^A-Z0-9 ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def distance(a, b):
    if a == b:
        return 0
    prec = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cour = [i]
        for j, cb in enumerate(b, 1):
            cour.append(min(prec[j] + 1, cour[j - 1] + 1, prec[j - 1] + (ca != cb)))
        prec = cour
    return prec[-1]


def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - m), min(1.0, c + m)


def ic(k, n):
    lo, hi = wilson(k, n)
    return f"[{lo * 100:.1f} ; {hi * 100:.1f}]"


def mcnemar(a_ok, b_ok):
    """(a seul juste, b seul juste, p exact)."""
    n10, n01 = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    return n10, n01, (stats.binomtest(n10, n10 + n01).pvalue if n10 + n01 else 1.0)


def manquant(v):
    return v is None or (isinstance(v, float) and math.isnan(v))


import s3fs
fs = s3fs.S3FileSystem()
print(f"mode {'TEST RAPIDE' if TEST_RAPIDE else 'COMPLET'}")

mode COMPLET


In [27]:
# 2. Le fichier des divergences arbitrees par le SeRN
def fichiers_divergences(c):
    try:
        return sorted(f for f in fs.find(f"{BUCKET}/data/raw/{c}/Divergences") if f.lower().endswith((".xlsx", ".xls")))
    except FileNotFoundError:
        return []

disponibles = {c: fichiers_divergences(c) for c in CAMPAGNES_POSSIBLES}
CAMPAGNE = CAMPAGNE or next((c for c in CAMPAGNES_POSSIBLES if disponibles[c]), None)
if not CAMPAGNE or not disponibles.get(CAMPAGNE):
    raise RuntimeError("Aucun fichier de divergences trouve sur S3.")
morceaux = []
for f in disponibles[CAMPAGNE]:
    with fs.open(f, "rb") as fh:
        morceaux.append(pd.read_excel(io.BytesIO(fh.read()), dtype=str))
DIV = pd.concat(morceaux, ignore_index=True).fillna("")
DIV.columns = [re.sub(r"[\s\-]+", "_", c.strip().upper()) for c in DIV.columns]
manque = [c for c in ["VARIABLE", "NOM_IMAGE", "VALEUR_PAD", "VALEUR_PMQ", "VALEUR_ATTENDUE"] if c not in DIV.columns]
if manque:
    raise RuntimeError(f"Colonnes introuvables dans le fichier des divergences : {manque}")

vals = DIV.VARIABLE.str.upper().str.strip().unique()
VARIABLE_NOM = VARIABLE_NOM or next((v for v in vals if re.fullmatch(r"(NOM|NOMI|NOM_INDIV|NOMNAIS)", v)), None)
VARIABLE_PRENOM = VARIABLE_PRENOM or next((v for v in vals if re.fullmatch(r"(PRENOMS?|PRENOMI|PRENOM_INDIV)", v)), None)
if not VARIABLE_NOM or not VARIABLE_PRENOM:
    print(DIV.VARIABLE.value_counts().head(15).to_string())
    raise RuntimeError("Variables NOM / PRENOM non reconnues : les renseigner en cellule 1.")


def cle_divergence(nom_image):
    """Les deux morceaux juste avant 'BI' : '/99/976/602/0407_9761_5300000184_BI.jpg' -> '9761_5300000184'."""
    m = Path(str(nom_image)).stem.split("_")
    if m and m[-1].upper() == "BI":
        m = m[:-1]
    return "_".join(m[-2:])


DIV["cle"] = DIV.NOM_IMAGE.map(cle_divergence)
DIV["champ"] = DIV.VARIABLE.str.upper().str.strip().map({VARIABLE_NOM: "NOM", VARIABLE_PRENOM: "PRENOM"})
DIV_TOUT = DIV.copy()
DIV = DIV[DIV.champ.notna()].copy()
DIV["arbitree"] = DIV.VALEUR_ATTENDUE.str.strip() != ""
print(f"campagne {CAMPAGNE} | {len(DIV_TOUT)} divergences, dont {len(DIV)} sur NOM/PRENOM "
      f"({int(DIV.arbitree.sum())} arbitrees) | NOM = {VARIABLE_NOM!r}, PRENOM = {VARIABLE_PRENOM!r}")

campagne RG25 | 149 divergences, dont 22 sur NOM/PRENOM (5 arbitrees) | NOM = 'NOM', PRENOM = 'PRENOM'


In [28]:
# 3. Les bulletins, le Fiqual et les images des blocs
fichiers = [f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/BIE") if f.endswith("_BI.jpg")]
bi = pd.DataFrame({"chemin": fichiers})
m = bi.chemin.map(lambda c: Path(c).stem).str.split("_")
bi["cle"] = m.str[0] + "_" + m.str[1]
fiq = [f for f in fs.ls(f"{BUCKET}/data/raw/{CAMPAGNE}") if "FIQUAL" in Path(f).name.upper() and f.lower().endswith(".txt")]
with fs.open(fiq[0], "rb") as f:
    brut = pd.read_csv(f, sep="\x00", header=None, names=["ligne"], dtype=str,
                       encoding="latin-1", keep_default_na=False, engine="python")["ligne"]
ch = brut.str.split("|")
fq3 = pd.DataFrame(ch[ch.str[0] == "3"].tolist(), dtype=str)
fq3["cle"] = fq3[POS_UT].str.strip() + "_" + fq3[POS_CAB].str.strip()
ref = fq3[["cle", COLS["NOM"], COLS["PRENOM"]]].copy()
ref.columns = ["cle", "NOM", "PRENOM"]
N_FIQUAL = ref.cle.nunique()
APP = bi.merge(ref, on="cle", how="inner").drop_duplicates("cle").sort_values("cle").reset_index(drop=True)
APP = APP.head(10) if TEST_RAPIDE else APP
print(f"Fiqual : {Path(fiq[0]).name} | {len(APP)} bulletins apparies | "
      f"divergences NOM/PRENOM sur nos bulletins : {int(DIV.cle.isin(set(APP.cle)).sum())} sur {len(DIV)}")

for cz in [RACINE / "configs" / f"positions_BI_{CAMPAGNE}.json", RACINE / "notebooks" / "configs" / f"positions_BI_{CAMPAGNE}.json"]:
    if cz.exists() and "bloc" in json.loads(cz.read_text(encoding="utf-8")):
        g, h, d_, b_ = json.loads(cz.read_text(encoding="utf-8"))["bloc"]
        ZONE_BLOC = (g, h, d_ - g, b_ - h)
        break

Fiqual : RP_PMQ_FIQUAL2599.txt | 169 bulletins apparies | divergences NOM/PRENOM sur nos bulletins : 22 sur 22


In [29]:
# 4. Le lecteur gemma4 (lectures du notebook 09 reutilisees si possible)
BASE_URL = "https://llm.lab.sspcloud.fr/api"
PROMPT_BLOC = """Cette image montre un extrait de bulletin individuel du recensement francais.
Elle contient des lignes manuscrites precedees de leurs libelles imprimes :
"Nom :" et "Prenom :".

Transcris ce qui est ecrit a la main sur chacune, en MAJUSCULES, sans accent.

Regles :
- Ne corrige rien, n'ajoute rien, ne devine pas.
- Ne recopie pas les libelles imprimes, seulement l'ecriture manuscrite.
- Si une ligne n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {"NOM": "...", "PRENOM": "..."}"""
PROMPT_RELECTURE = """Cette image montre un extrait de bulletin individuel du recensement francais,
avec un nom et un prenom ecrits a la main apres les libelles imprimes "Nom :" et "Prenom :".

Une premiere lecture a donne : {lecture}
Ces valeurs ne figurent pas dans les listes de l'Insee des noms et prenoms portes en France.
Voici des valeurs existantes, proches de la premiere lecture :
{candidats}

Relis attentivement l'ecriture manuscrite.
- Si elle correspond vraiment a l'une de ces valeurs, reponds cette valeur.
- Sinon, recopie exactement ce qui est ecrit, meme si ce n'est pas dans la liste.
- Si ce n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {{"NOM": "...", "PRENOM": "..."}}"""
HEADERS = None


def service_pret():
    global HEADERS
    if HEADERS:
        return
    if "OPENAI_API_KEY" not in os.environ and shutil.which("vault"):
        r = subprocess.run(["vault", "kv", "get", "-field=OPENAI_API_KEY", "onyxia-kv/farisazibert/LLM"],
                           capture_output=True, text=True, timeout=60)
        if r.returncode == 0 and r.stdout.strip():
            os.environ["OPENAI_API_KEY"] = r.stdout.strip()
    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("Cle du service absente : la renseigner (Vault).")
    HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}


def bloc(cle):
    fb = BLOCS / f"{CAMPAGNE}_{cle}.png"
    if not fb.exists():
        chemin = APP.loc[APP.cle == cle, "chemin"].iloc[0]
        with fs.open(chemin, "rb") as src:
            page = Image.open(io.BytesIO(src.read())).convert("RGB")
        L, H = page.size
        x, y, w, hh = ZONE_BLOC
        page.crop((int(x * L), int(y * H), int((x + w) * L), int((y + hh) * H))).save(fb)
    return Image.open(fb)


def lire(image, consigne, tentatives=3):
    service_pret()
    t = io.BytesIO()
    image.convert("RGB").save(t, format="PNG")
    b64 = base64.b64encode(t.getvalue()).decode()
    for essai in range(tentatives):
        try:
            r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, timeout=180, json={
                "model": MODELE, "temperature": 0, "max_tokens": 200,
                "messages": [{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {"type": "text", "text": consigne}]}]})
            r.raise_for_status()
            mm = re.search(r"\{.*\}", r.json()["choices"][0]["message"]["content"] or "", re.S)
            dd = json.loads(mm.group(0)) if mm else {}
            return {c: str(dd.get(c, "") or "") for c in COLS}
        except Exception:
            time.sleep(2 * (essai + 1))
    return {c: "" for c in COLS}


def lectures(cache, cles, fabrique):
    """Lit (ou relit depuis le cache) un ensemble de bulletins ; renvoie cle, champ, lu."""
    L_ = pd.read_csv(cache, dtype=str).fillna("") if cache.exists() else pd.DataFrame(columns=["cle", "champ", "lu"])
    faire = [c for c in cles if c not in set(L_.cle)]
    rows, t0 = [], time.time()
    for k, cle in enumerate(faire):
        res = fabrique(cle)
        rows += [{"cle": cle, "champ": c, "lu": res[c]} for c in COLS]
        if (k + 1) % 25 == 0:
            print(f"  {k + 1}/{len(faire)} | {(time.time() - t0) / 60:.1f} min", flush=True)
    if rows:
        L_ = pd.concat([L_, pd.DataFrame(rows)], ignore_index=True)
        L_.to_csv(cache, index=False)
    return L_[L_.cle.isin(set(cles))]


cache_base = RESULTATS / ("listes_insee_base.csv" if CAMPAGNE == "RG25" else f"arbitrage_{CAMPAGNE}_lectures.csv")
LECT = lectures(cache_base, list(APP.cle), lambda cle: lire(bloc(cle), PROMPT_BLOC))
print(f"lectures gemma4 : {LECT.cle.nunique()} bulletins ({cache_base.name})")

lectures gemma4 : 169 bulletins (listes_insee_base.csv)


In [30]:
# 5. Les listes de l'Insee et les 6 usages (memes regles que le notebook 09)
LISTE = {}
for f, champ in [("noms2008nat_txt.zip", "NOM"), ("prenoms-2025-nat_csv.zip", "PRENOM")]:
    with zipfile.ZipFile(LISTES / f) as z:
        brut_l = z.read(next(n for n in z.namelist() if n.lower().endswith((".txt", ".csv"))))
    for e in ("utf-8", "latin-1"):
        try:
            texte = brut_l.decode(e)
            break
        except UnicodeDecodeError:
            continue
    sep = max(["\t", ";", ","], key=texte.split("\n", 1)[0].count)
    df = pd.read_csv(io.StringIO(texte), sep=sep, dtype=str, keep_default_na=False)
    df.columns = [x.strip().upper() for x in df.columns]
    if champ == "NOM":
        col = df.columns[0]
        n = df.drop(columns=[col]).apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
        dl = pd.DataFrame({"v": df[col].map(normaliser), "n": n})
        dl = dl[(dl.v != "") & (dl.v != "AUTRES NOMS")]
    else:
        col = next(x for x in ["PRENOM", "PREUSUEL"] if x in df.columns)
        colv = next(x for x in ["VALEUR", "NOMBRE"] if x in df.columns)
        dl = pd.DataFrame({"v": df[col].map(normaliser), "n": pd.to_numeric(df[colv], errors="coerce").fillna(0)})
        dl = dl[(dl.v != "") & ~dl.v.str.contains("PRENOMS RARES")]
    LISTE[champ] = dl.groupby("v").n.sum().to_dict()

ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ "


def dans_liste(v, c):
    if not v or v == "ILLISIBLE":
        return True
    return v in LISTE[c] or all(x in LISTE[c] for x in v.split())


def voisins1(mot):
    coupes = [(mot[:i], mot[i:]) for i in range(len(mot) + 1)]
    return set([a + b[1:] for a, b in coupes if b] + [a + b[1] + b[0] + b[2:] for a, b in coupes if len(b) > 1]
               + [a + x + b[1:] for a, b in coupes if b for x in ALPHABET] + [a + x + b for a, b in coupes for x in ALPHABET])


def candidats(v, c, ecart_max=1):
    v1 = voisins1(v)
    t = {x for x in v1 if x in LISTE[c]}
    if ecart_max >= 2 and len(t) < 5:
        for x in list(v1):
            t |= {w for w in voisins1(x) if w in LISTE[c]}
    t.discard(v)
    return sorted(t, key=lambda x: -LISTE[c][x])


def corriger(v, c, regle):
    if dans_liste(v, c) or len(v) < LONGUEUR_MIN_CORRECTION:
        return v
    k = candidats(v, c)
    if regle == "unique":
        return k[0] if len(k) == 1 else v
    return k[0] if k else v


LU = LECT.copy()
LU["gemma4"] = LU.lu.map(normaliser)
LU["absent_liste"] = [int(not dans_liste(v, c)) for v, c in zip(LU.gemma4, LU.champ)]

# Usage 3 : relecture des seuls blocs suspects, avec des candidats
suspects = sorted(LU[LU.absent_liste == 1].cle.unique())
def relecture(cle):
    lignes = LU[LU.cle == cle].set_index("champ")
    lecture = ", ".join(f"{c} = {lignes.loc[c, 'gemma4'] or 'vide'}" for c in COLS if c in lignes.index)
    props = [f"- {c} : {', '.join(candidats(lignes.loc[c, 'gemma4'], c, 2)[:5]) or '(aucune valeur proche)'}"
             for c in COLS if c in lignes.index and lignes.loc[c, "absent_liste"] == 1]
    return lire(bloc(cle), PROMPT_RELECTURE.format(lecture=lecture, candidats="\n".join(props)))
cache_rel = RESULTATS / ("listes_insee_relecture.csv" if CAMPAGNE == "RG25" else f"arbitrage_{CAMPAGNE}_relecture.csv")
REL = lectures(cache_rel, suspects, relecture)
rel = {(r.cle, r.champ): normaliser(r.lu) for r in REL.itertuples()}

LU["correction_unique"] = [corriger(v, c, "unique") for v, c in zip(LU.gemma4, LU.champ)]
LU["correction_frequent"] = [corriger(v, c, "frequent") for v, c in zip(LU.gemma4, LU.champ)]
LU["consigne"] = [rel.get((k, c), v) if a else v for k, c, v, a in zip(LU.cle, LU.champ, LU.gemma4, LU.absent_liste)]
# Usage 4 : apres la consigne, ce qui RESTE absent des listes (ou illisible) part en relecture humaine
LU["absent_apres_consigne"] = [int(v in ("", "ILLISIBLE") or not dans_liste(v, c)) for v, c in zip(LU.consigne, LU.champ)]
LU["candidats_insee"] = [", ".join(candidats(v, c, 2)[:5]) if a else "" for v, c, a in zip(LU.gemma4, LU.champ, LU.absent_liste)]
print(f"listes : {len(LISTE['NOM'])} noms, {len(LISTE['PRENOM'])} prenoms | "
      f"{len(suspects)} blocs suspects relus (consigne) | lignes encore suspectes apres consigne : "
      f"{int(LU.absent_apres_consigne.sum())} (contre {int(LU.absent_liste.sum())} avant)")

listes : 218114 noms, 43449 prenoms | 41 blocs suspects relus (consigne) | lignes encore suspectes apres consigne : 37 (contre 47 avant)


In [31]:
# 6. La table ligne par ligne : PMQ, PAD, SeRN, verite arbitree, et les 6 usages
divx = {(r.cle, r.champ): r for r in DIV.itertuples()}
fiq_val = {(r.cle, c): normaliser(getattr(r, c)) for r in APP.itertuples() for c in COLS}
T = LU.copy()
T["PMQ"] = [fiq_val.get((k, c), "") for k, c in zip(T.cle, T.champ)]
T = T[T.PMQ != ""].reset_index(drop=True)
statut, pad, sern, verite = [], [], [], []
for k, c, p in zip(T.cle, T.champ, T.PMQ):
    d = divx.get((k, c))
    if d is None:
        statut.append("sans divergence"); pad.append(p); sern.append(""); verite.append(p)
    elif not d.arbitree:
        statut.append("divergence non arbitree"); pad.append(normaliser(d.VALEUR_PAD)); sern.append(""); verite.append(None)
    else:
        statut.append("divergence arbitree"); pad.append(normaliser(d.VALEUR_PAD))
        sern.append(normaliser(d.VALEUR_ATTENDUE)); verite.append(normaliser(d.VALEUR_ATTENDUE))
T["statut"], T["PAD"], T["SeRN"], T["verite_arbitree"] = statut, pad, sern, verite
T["groupe"] = T.statut.map({"sans divergence": "A", "divergence arbitree": "B", "divergence non arbitree": "C"})

# usage : (colonne lue, colonne qui dit « envoye en relecture humaine » ou None)
USAGES = {"gemma4 seul": ("gemma4", None),
          "1. signal (relecture des absents)": ("gemma4", "absent_liste"),
          "2a. correction, candidat unique": ("correction_unique", None),
          "2b. correction, le plus frequent": ("correction_frequent", None),
          "3. consigne avec candidats": ("consigne", None),
          "4. consigne + signal": ("consigne", "absent_apres_consigne")}


def case(valeur, vrai, releve=False):
    if manquant(vrai):
        return "ecartee"
    if valeur == vrai:
        return "juste"
    if valeur in ("", "ILLISIBLE"):
        return "avouee"
    return "relue" if releve else "silencieuse"


for nom, (col, sig) in USAGES.items():
    rel_ = T[sig].astype(bool) if sig else pd.Series(False, index=T.index)
    T[f"{nom} | Fiqual"] = [case(v, t, r) for v, t, r in zip(T[col], T.PMQ, rel_)]
    T[f"{nom} | arbitree"] = [case(v, t, r) for v, t, r in zip(T[col], T.verite_arbitree, rel_)]
T["PMQ | arbitree"] = [case(v, t) for v, t in zip(T.PMQ, T.verite_arbitree)]
T["PAD | arbitree"] = [case(v, t) for v, t in zip(T.PAD, T.verite_arbitree)]
print(f"{len(T)} lignes | " + " | ".join(f"groupe {g} ({s}) : {n}" for (g, s), n in
                                         T.groupby(["groupe", "statut"]).size().items()))

335 lignes | groupe A (sans divergence) : 313 | groupe B (divergence arbitree) : 5 | groupe C (divergence non arbitree) : 17


In [32]:
# 7. TABLEAU PRINCIPAL : chaque usage juge par rapport au Fiqual ET a la verite arbitree
arb = T.verite_arbitree.map(lambda v: not manquant(v))
pmq_ok = (T.PMQ == T.verite_arbitree)[arb]


def a_un_humain(nom):
    """Lignes envoyees a un humain : signalees par l'usage, ou rendues vides / ILLISIBLE."""
    if nom not in USAGES:
        return None
    col, sig = USAGES[nom]
    h_ = T[col].isin(["", "ILLISIBLE"])
    return h_ | T[sig].astype(bool) if sig else h_


def resume(colonne, nom, verite, comparer=None):
    s = T[colonne]
    s = s[s != "ecartee"]
    n, k = len(s), int((s == "juste").sum())
    hum = a_un_humain(nom)
    ligne = {"usage": nom, "verite": verite, "N": n, "justes": k, "justes_%": round(k / n * 100, 1),
             "IC95": ic(k, n), "avouees_%": round((s == "avouee").mean() * 100, 1),
             "relues_%": round((s == "relue").mean() * 100, 1),
             "silencieuses": int((s == "silencieuse").sum()),
             "silencieuses_%": round((s == "silencieuse").mean() * 100, 1),
             "a_un_humain_%": round(hum.loc[s.index].mean() * 100, 1) if hum is not None else 100.0}
    if comparer is not None:
        a, b, p = mcnemar((s == "juste"), comparer.loc[s.index])
        ligne["vs PMQ (seul juste / PMQ seul juste)"] = f"{a} / {b}"
        ligne["McNemar p"] = round(p, 4)
    return ligne


lignes = []
for nom in USAGES:
    lignes.append(resume(f"{nom} | Fiqual", nom, "Fiqual"))
    lignes.append(resume(f"{nom} | arbitree", nom, "arbitree", pmq_ok))
lignes += [resume("PMQ | arbitree", "PMQ (saisie humaine)", "arbitree"),
           resume("PAD | arbitree", "PAD (saisie humaine)", "arbitree")]
SYNTHESE = pd.DataFrame(lignes)
with pd.option_context("display.max_columns", 14, "display.width", 250):
    display(SYNTHESE)
print("\nPour chaque ligne : justes + avouees + relues + silencieuses = 100 %."
      "\n'relues' = fausses mais envoyees en relecture humaine (usages 1 et 4 seulement) : un humain les corrigera."
      "\nverite 'arbitree' : les divergences non arbitrees sont ecartees, d'ou un N plus petit."
      "\na_un_humain_% = part des lignes qu'un humain doit relire (signalees, vides ou ILLISIBLE) : c'est le COUT."
      "\nvs PMQ : sur les memes lignes, combien cet usage a juste seul, et combien le PMQ a juste seul.")

,usage,verite,N,justes,justes_%,IC95,avouees_%,relues_%,silencieuses,silencieuses_%,a_un_humain_%,vs PMQ (seul juste / PMQ seul juste),McNemar p
0,gemma4 seul,Fiqual,335,281,83.9,[79.6 ; 87.4],0.9,0.0,51,15.2,0.9,NaN,NaN
1,gemma4 seul,arbitree,318,280,88.1,[84.0 ; 91.2],0.0,0.0,38,11.9,0.0,3 / 36,0.0
2,1. signal (relecture des absents),Fiqual,335,281,83.9,[79.6 ; 87.4],0.9,7.5,26,7.8,14.9,NaN,NaN
3,1. signal (relecture des absents),arbitree,318,280,88.1,[84.0 ; 91.2],0.0,6.0,19,6.0,12.6,3 / 36,0.0
4,"2a. correction, candidat unique",Fiqual,335,281,83.9,[79.6 ; 87.4],0.9,0.0,51,15.2,0.9,NaN,NaN
5,"2a. correction, candidat unique",arbitree,318,280,88.1,[84.0 ; 91.2],0.0,0.0,38,11.9,0.0,3 / 36,0.0
6,"2b. correction, le plus frequent",Fiqual,335,274,81.8,[77.3 ; 85.6],0.9,0.0,58,17.3,0.9,NaN,NaN
7,"2b. correction, le plus frequent",arbitree,318,271,85.2,[80.9 ; 88.7],0.0,0.0,47,14.8,0.0,3 / 45,0.0
8,3. consigne avec candidats,Fiqual,335,290,86.6,[82.5 ; 89.8],0.9,0.0,42,12.5,0.9,NaN,NaN
9,3. consigne avec candidats,arbitree,318,288,90.6,[86.9 ; 93.3],0.0,0.0,30,9.4,0.0,3 / 28,0.0



Pour chaque ligne : justes + avouees + relues + silencieuses = 100 %.
'relues' = fausses mais envoyees en relecture humaine (usages 1 et 4 seulement) : un humain les corrigera.
verite 'arbitree' : les divergences non arbitrees sont ecartees, d'ou un N plus petit.
a_un_humain_% = part des lignes qu'un humain doit relire (signalees, vides ou ILLISIBLE) : c'est le COUT.
vs PMQ : sur les memes lignes, combien cet usage a juste seul, et combien le PMQ a juste seul.


In [33]:
# 8. DECOMPOSITION EN TROIS GROUPES : d'ou viennent les « justes » ?
#   A = pas de divergence (PAD = PMQ)  -> Fiqual et verite arbitree DISENT LA MEME CHOSE
#   B = divergence arbitree par le SeRN -> la verite arbitree peut contredire le Fiqual
#   C = divergence non arbitree         -> verite inconnue, ligne ecartee de la verite arbitree
taille = T.groupe.value_counts().reindex(["A", "B", "C"]).fillna(0).astype(int)
pmq_B = int(((T.groupe == "B") & (T["PMQ | arbitree"] == "juste")).sum())
print(f"Groupe A : {taille['A']} lignes | groupe B : {taille['B']} lignes (le PMQ y a raison {pmq_B} fois) | "
      f"groupe C : {taille['C']} lignes\n")

dec = []
for nom in USAGES:
    F, A_ = T[f"{nom} | Fiqual"] == "juste", T[f"{nom} | arbitree"] == "juste"
    gA, gB, gC = T.groupe == "A", T.groupe == "B", T.groupe == "C"
    dec.append({"usage": nom,
                "justes Fiqual (/ %d)" % len(T): int(F.sum()),
                "dont A : vraiment justes": int((F & gA).sum()),
                "dont B : confirmes par le SeRN": int((F & gB & A_).sum()),
                "dont B : en fait FAUX (copie l'erreur du PMQ)": int((F & gB & ~A_).sum()),
                "dont C : on ne sait pas": int((F & gC).sum()),
                "faux Fiqual mais JUSTES (raison contre le PMQ)": int((~F & gB & A_).sum()),
                "justes arbitree (/ %d)" % int((T.groupe != "C").sum()): int(A_.sum()),
                "faux Fiqual dans C (on ne sait pas)": int((~F & gC).sum())})
DECOMPOSITION = pd.DataFrame(dec)
with pd.option_context("display.max_columns", 12, "display.width", 250):
    display(DECOMPOSITION.set_index("usage").T)
print("Verification : justes arbitree = A + B confirmes + faux Fiqual mais justes.")
for r in dec:
    v = list(r.values())
    assert v[7] == v[2] + v[3] + v[6], r["usage"]
print("OK pour tous les usages.")

Groupe A : 313 lignes | groupe B : 5 lignes (le PMQ y a raison 0 fois) | groupe C : 17 lignes



usage,gemma4 seul,1. signal (relecture des absents),"2a. correction, candidat unique","2b. correction, le plus frequent",3. consigne avec candidats,4. consigne + signal
justes Fiqual (/ 335),281,281,281,274,290,290
dont A : vraiment justes,277,277,277,268,285,285
dont B : confirmes par le SeRN,0,0,0,0,0,0
dont B : en fait FAUX (copie l'erreur du PMQ),0,0,0,0,0,0
dont C : on ne sait pas,4,4,4,6,5,5
faux Fiqual mais JUSTES (raison contre le PMQ),3,3,3,3,3,3
justes arbitree (/ 318),280,280,280,271,288,288
faux Fiqual dans C (on ne sait pas),13,13,13,11,12,12


Verification : justes arbitree = A + B confirmes + faux Fiqual mais justes.
OK pour tous les usages.


In [34]:
# 9. TABLEAU CROISE : les « justes » par rapport au Fiqual le sont-ils vraiment ?
for nom in ["gemma4 seul", "3. consigne avec candidats", "4. consigne + signal"]:
    x = pd.crosstab(T[f"{nom} | Fiqual"].map(lambda v: "juste" if v == "juste" else "faux"),
                    T[f"{nom} | arbitree"].map(lambda v: {"juste": "juste", "ecartee": "ecartee"}.get(v, "faux")),
                    margins=True, margins_name="total")
    x.index.name, x.columns.name = "par rapport au Fiqual", "par rapport a la verite arbitree"
    print(f"\n=== {nom} ===")
    display(x)
    jj = int(((T[f"{nom} | Fiqual"] == "juste") & (T[f"{nom} | arbitree"] == "juste")).sum())
    jf = int(((T[f"{nom} | Fiqual"] == "juste") & T[f"{nom} | arbitree"].isin(["silencieuse", "avouee", "relue"])).sum())
    fj = int(((T[f"{nom} | Fiqual"] != "juste") & (T[f"{nom} | arbitree"] == "juste")).sum())
    je = int(((T[f"{nom} | Fiqual"] == "juste") & (T[f"{nom} | arbitree"] == "ecartee")).sum())
    print(f"- {jj} lignes justes dans les deux cas : confirmees.")
    print(f"- {jf} lignes comptees justes par le Fiqual sont en fait FAUSSES : la lecture etait identique "
          f"a celle du PMQ, et le PMQ s'etait trompe.")
    print(f"- {fj} lignes comptees fausses par le Fiqual sont en fait JUSTES : la lecture avait raison contre le PMQ.")
    print(f"- {je} lignes justes par rapport au Fiqual tombent sur une divergence non arbitree : on ne sait pas.")


=== gemma4 seul ===


par rapport a la verite arbitree,ecartee,faux,juste,total
par rapport au Fiqual,,,,
faux,13,38,3,54
juste,4,0,277,281
total,17,38,280,335


- 277 lignes justes dans les deux cas : confirmees.
- 0 lignes comptees justes par le Fiqual sont en fait FAUSSES : la lecture etait identique a celle du PMQ, et le PMQ s'etait trompe.
- 3 lignes comptees fausses par le Fiqual sont en fait JUSTES : la lecture avait raison contre le PMQ.
- 4 lignes justes par rapport au Fiqual tombent sur une divergence non arbitree : on ne sait pas.

=== 3. consigne avec candidats ===


par rapport a la verite arbitree,ecartee,faux,juste,total
par rapport au Fiqual,,,,
faux,12,30,3,45
juste,5,0,285,290
total,17,30,288,335


- 285 lignes justes dans les deux cas : confirmees.
- 0 lignes comptees justes par le Fiqual sont en fait FAUSSES : la lecture etait identique a celle du PMQ, et le PMQ s'etait trompe.
- 3 lignes comptees fausses par le Fiqual sont en fait JUSTES : la lecture avait raison contre le PMQ.
- 5 lignes justes par rapport au Fiqual tombent sur une divergence non arbitree : on ne sait pas.

=== 4. consigne + signal ===


par rapport a la verite arbitree,ecartee,faux,juste,total
par rapport au Fiqual,,,,
faux,12,30,3,45
juste,5,0,285,290
total,17,30,288,335


- 285 lignes justes dans les deux cas : confirmees.
- 0 lignes comptees justes par le Fiqual sont en fait FAUSSES : la lecture etait identique a celle du PMQ, et le PMQ s'etait trompe.
- 3 lignes comptees fausses par le Fiqual sont en fait JUSTES : la lecture avait raison contre le PMQ.
- 5 lignes justes par rapport au Fiqual tombent sur une divergence non arbitree : on ne sait pas.


In [35]:
# 10. ANALYSE DES DIVERGENCES : qui avait raison, et que fait gemma4 sur ces lignes ?
DV = T[T.statut != "sans divergence"].copy()
DV["PMQ_juste"] = [("?" if manquant(v) else ("oui" if p == v else "non")) for p, v in zip(DV.PMQ, DV.verite_arbitree)]
DV["PAD_juste"] = [("?" if manquant(v) else ("oui" if p == v else "non")) for p, v in zip(DV.PAD, DV.verite_arbitree)]
DV["gemma4_rejoint"] = ["SeRN" if (not manquant(v) and g == v) else "PMQ" if g == p else "PAD" if g == a
                        else ("illisible" if g in ("", "ILLISIBLE") else "aucun")
                        for g, p, a, v in zip(DV.gemma4, DV.PMQ, DV.PAD, DV.verite_arbitree)]

print(f"{len(DV)} divergences PAD / PMQ sur NOM et PRENOM, soit {len(DV) / len(T):.1%} des lignes.\n")
print("Qui avait raison selon le SeRN ?")
display(pd.crosstab(DV.champ, [DV.PMQ_juste.rename("PMQ juste"), DV.PAD_juste.rename("PAD juste")], margins=True))
print("\nQue lit gemma4 sur ces lignes ?")
display(pd.crosstab(DV.statut, DV.gemma4_rejoint, margins=True))
DETAIL_DIV = DV[["cle", "champ", "groupe", "statut", "PMQ", "PAD", "SeRN", "lu", "gemma4", "consigne", "absent_liste",
                 "PMQ_juste", "PAD_juste", "gemma4_rejoint"]]
with pd.option_context("display.max_rows", 80, "display.max_columns", 14, "display.width", 250):
    display(DETAIL_DIV)

22 divergences PAD / PMQ sur NOM et PRENOM, soit 6.6% des lignes.

Qui avait raison selon le SeRN ?


PMQ juste,?,non,All
PAD juste,?,oui,
champ,,,
NOM,9,5,14
PRENOM,8,0,8
All,17,5,22



Que lit gemma4 sur ces lignes ?


gemma4_rejoint,PAD,PMQ,SeRN,aucun,illisible,All
statut,,,,,,
divergence arbitree,0,0,3,2,0,5
divergence non arbitree,3,4,0,7,3,17
All,3,4,3,9,3,22


,cle,champ,groupe,statut,PMQ,PAD,SeRN,lu,gemma4,consigne,absent_liste,PMQ_juste,PAD_juste,gemma4_rejoint
4,6901_5300000222,NOM,B,divergence arbitree,PEREZ,PEREZ JEAN,PEREZ JEAN,PEREZ JEAN,PEREZ JEAN,PEREZ JEAN,0,non,oui,SeRN
5,6901_5300000222,PRENOM,C,divergence non arbitree,JEAN,,,ILLISIBLE,ILLISIBLE,ILLISIBLE,0,?,?,illisible
13,6901_5300000298,NOM,B,divergence arbitree,DUPONT,DUNANT,DUNANT,DUNANT,DUNANT,DUNANT,0,non,oui,SeRN
41,6901_5300000360,NOM,C,divergence non arbitree,JUNOU,JUNON,,JUNON,JUNON,JUNON,0,?,?,PAD
53,6901_5300000363,NOM,C,divergence non arbitree,AUDEN CHRIST,AUDEM CHRISTIAN,,AUDENOT CHRETIEN,AUDENOT CHRETIEN,AUDENOT CHRETIEN,0,?,?,aucun
74,6901_5300000401,NOM,B,divergence arbitree,BARZIC,BARZIL ANNE,BARZIL ANNE,BARZIC ANNE,BARZIC ANNE,BARZIC ANNE,0,non,oui,aucun
75,6901_5300000401,PRENOM,C,divergence non arbitree,ANNE,,,ILLISIBLE,ILLISIBLE,ILLISIBLE,0,?,?,illisible
82,6901_5300000424,NOM,B,divergence arbitree,DUVAL,DUVAL HELENE,DUVAL HELENE,DUVAL HELENE,DUVAL HELENE,DUVAL HELENE,0,non,oui,SeRN
83,6901_5300000424,PRENOM,C,divergence non arbitree,HELENE,,,ILLISIBLE,ILLISIBLE,ILLISIBLE,0,?,?,illisible
110,7601_5300000265,NOM,C,divergence non arbitree,VION,VON,,VION,VION,VION,0,?,?,PMQ


In [36]:
# 11. TYPOLOGIE DES ERREURS de gemma4 (verite arbitree) : quel genre, et qui les attrape
def type_erreur(lu, vrai):
    if lu in ("", "ILLISIBLE"):
        return "illisible ou vide"
    if lu.replace(" ", "") == vrai.replace(" ", ""):
        return "espace ou tiret"
    mots_lu, mots_vrai = set(lu.split()), set(vrai.split())
    if mots_lu < mots_vrai or mots_vrai < mots_lu:
        return "nom compose incomplet ou en trop"
    dd = distance(lu, vrai)
    if dd == 1:
        return "1 lettre"
    if dd == 2:
        return "2 lettres"
    return "autre (3 lettres ou plus)"


E = T[T["gemma4 seul | arbitree"].isin(["silencieuse", "avouee"])].copy()
E["type"] = [type_erreur(l, v) for l, v in zip(E.gemma4, E.verite_arbitree)]
E["lettres_fausses"] = [distance(l, v) for l, v in zip(E.gemma4, E.verite_arbitree)]
E["signal"] = E["1. signal (relecture des absents) | arbitree"]
E["apres_consigne"] = E["3. consigne avec candidats | arbitree"]
E["consigne_signal"] = E["4. consigne + signal | arbitree"]
TYPES = (E.groupby("type").agg(erreurs=("cle", "size"),
                               dont_NOM=("champ", lambda s: int((s == "NOM").sum())),
                               attrapees_signal=("signal", lambda s: int((s == "relue").sum())),
                               reparees_consigne=("apres_consigne", lambda s: int((s == "juste").sum())),
                               rattrapees_consigne_signal=("consigne_signal",
                                                           lambda s: int(s.isin(["juste", "relue"]).sum())))
         .sort_values("erreurs", ascending=False))
TYPES["part_%"] = (TYPES.erreurs / TYPES.erreurs.sum() * 100).round(1)
display(TYPES)
print(f"{len(E)} erreurs de gemma4 sur {int((T['gemma4 seul | arbitree'] != 'ecartee').sum())} lignes.")
print("rattrapees_consigne_signal = reparees par la consigne OU envoyees a un humain par le signal.")
print("Les erreurs d'une ou deux lettres produisent souvent un nom inexistant (attrape par le signal) ;"
      "\nles confusions entre deux noms reels ('autre') echappent a la liste.")
ERREURS = E[["cle", "champ", "groupe", "verite_arbitree", "lu", "gemma4", "type", "lettres_fausses", "absent_liste",
             "consigne", "signal", "apres_consigne", "consigne_signal"]]

,erreurs,dont_NOM,attrapees_signal,reparees_consigne,rattrapees_consigne_signal,part_%
type,,,,,,
1 lettre,22,14,10,3,9,57.9
2 lettres,10,5,7,5,6,26.3
autre (3 lettres ou plus),4,1,1,0,1,10.5
espace ou tiret,1,1,1,1,1,2.6
nom compose incomplet ou en trop,1,0,0,0,0,2.6


38 erreurs de gemma4 sur 318 lignes.
rattrapees_consigne_signal = reparees par la consigne OU envoyees a un humain par le signal.
Les erreurs d'une ou deux lettres produisent souvent un nom inexistant (attrape par le signal) ;
les confusions entre deux noms reels ('autre') echappent a la liste.


In [37]:
# 12. LA BARRE : la fourchette du taux d'erreur du PMQ, et ou se situe chaque usage
n_tot = len(T)
n_err_pmq = int((T["PMQ | arbitree"] == "silencieuse").sum())
n_non_arb = int((T.statut == "divergence non arbitree").sum())
bas, haut = n_err_pmq / (n_tot - n_non_arb), (n_err_pmq + n_non_arb) / n_tot
print(f"Taux d'erreur du PMQ sur nos {n_tot} lignes :")
print(f"- au minimum {bas:.1%} : {n_err_pmq} erreurs prouvees par le SeRN ;")
print(f"- au maximum {haut:.1%} : si le PMQ avait tort sur les {n_non_arb} divergences non arbitrees ;")
print("- (et un peu plus si le PAD et le PMQ se sont trompes de la meme facon : invisible ici).\n")

BARRE = SYNTHESE[SYNTHESE.verite == "arbitree"][["usage", "N", "justes_%", "silencieuses_%", "a_un_humain_%"]].copy()
BARRE["position"] = ["sous la barre basse du PMQ" if s / 100 <= bas else
                     "dans la fourchette du PMQ" if s / 100 <= haut else "au-dessus du PMQ"
                     for s in BARRE["silencieuses_%"]]
display(BARRE)
print("Note : la ligne PAD ne repose que sur les divergences arbitrees ; ne pas la citer comme un taux d'erreur du PAD.")
print("Critere : les erreurs SILENCIEUSES de l'usage, comparees au taux d'erreur du PMQ "
      "(pour un humain, toutes les erreurs sont silencieuses).")

# Contexte : toute la campagne, pas seulement nos bulletins
tout = DIV[DIV.arbitree]
CONTEXTE = pd.DataFrame([{"champ": c, "divergences_arbitrees": int((tout.champ == c).sum()),
                          "erreurs_PMQ": int((tout[tout.champ == c].VALEUR_PMQ.map(normaliser)
                                             != tout[tout.champ == c].VALEUR_ATTENDUE.map(normaliser)).sum()),
                          "erreurs_PAD": int((tout[tout.champ == c].VALEUR_PAD.map(normaliser)
                                             != tout[tout.champ == c].VALEUR_ATTENDUE.map(normaliser)).sum())}
                         for c in COLS])
print(f"\nSur tout le fichier des divergences ({CAMPAGNE}) :")
display(CONTEXTE)

Taux d'erreur du PMQ sur nos 335 lignes :
- au minimum 1.6% : 5 erreurs prouvees par le SeRN ;
- au maximum 6.6% : si le PMQ avait tort sur les 17 divergences non arbitrees ;
- (et un peu plus si le PAD et le PMQ se sont trompes de la meme facon : invisible ici).



,usage,N,justes_%,silencieuses_%,a_un_humain_%,position
1,gemma4 seul,318,88.1,11.9,0.0,au-dessus du PMQ
3,1. signal (relecture des absents),318,88.1,6.0,12.6,dans la fourchette du PMQ
5,"2a. correction, candidat unique",318,88.1,11.9,0.0,au-dessus du PMQ
7,"2b. correction, le plus frequent",318,85.2,14.8,0.0,au-dessus du PMQ
9,3. consigne avec candidats,318,90.6,9.4,0.0,au-dessus du PMQ
11,4. consigne + signal,318,90.6,6.9,9.1,au-dessus du PMQ
12,PMQ (saisie humaine),318,98.4,1.6,100.0,dans la fourchette du PMQ
13,PAD (saisie humaine),318,100.0,0.0,100.0,sous la barre basse du PMQ


Note : la ligne PAD ne repose que sur les divergences arbitrees ; ne pas la citer comme un taux d'erreur du PAD.
Critere : les erreurs SILENCIEUSES de l'usage, comparees au taux d'erreur du PMQ (pour un humain, toutes les erreurs sont silencieuses).

Sur tout le fichier des divergences (RG25) :


,champ,divergences_arbitrees,erreurs_PMQ,erreurs_PAD
0,NOM,5,5,0
1,PRENOM,0,0,0


In [38]:
# 13. FICHIER DE DETAIL LIGNE PAR LIGNE, avec l'image du bloc a cote de chaque ligne a regarder
#     -> arbitrage_{CAMPAGNE}_detail_lignes.xlsx (Excel, images incluses)
#     -> arbitrage_{CAMPAGNE}_galerie.html       (meme contenu, a ouvrir dans le navigateur)
MAX_IMAGES = 400                 # au-dela, pas d'image (le fichier deviendrait trop lourd)
LARGEUR_IMAGE = 460              # pixels


def verdict(v, vrai):
    if manquant(vrai):
        return "VERITE INCONNUE"
    if v == vrai:
        return "JUSTE"
    if v == "":
        return "VIDE (rien lu)"
    if v == "ILLISIBLE":
        return "ILLISIBLE (avoue)"
    return "TROMPE"


def contradiction(f, a):
    if f == "juste" and a == "ecartee":
        return "juste Fiqual -> inconnu (divergence non arbitree)"
    if f == "juste" and a != "juste":
        return "juste Fiqual -> FAUX en verite (copie l'erreur du PMQ)"
    if f != "juste" and a == "juste":
        return "faux Fiqual -> JUSTE en verite (raison contre le PMQ)"
    return ""


def commentaire(r):
    morceaux = []
    if r.groupe == "B":
        qui = "le PMQ" if r.PMQ == r.verite_arbitree else ("le PAD" if r.PAD == r.verite_arbitree else "ni l'un ni l'autre")
        morceaux.append(f"PAD '{r.PAD}' et PMQ '{r.PMQ}' divergent ; le SeRN tranche '{r.SeRN}' (raison : {qui})")
    elif r.groupe == "C":
        morceaux.append(f"PAD '{r.PAD}' et PMQ '{r.PMQ}' divergent ; pas d'arbitrage : verite inconnue")
    vg = r.verdict_gemma4
    vrai = r.PMQ if vg == "VERITE INCONNUE" else r.verite_arbitree
    if vg == "TROMPE":
        morceaux.append(f"gemma4 lit '{r.gemma4}' au lieu de '{vrai}' ({r.type_erreur})")
    elif vg.startswith("VIDE") or vg.startswith("ILLISIBLE"):
        morceaux.append(f"gemma4 ne donne rien ({vg.lower()}) ; attendu '{vrai}'")
    elif vg == "VERITE INCONNUE":
        morceaux.append(f"gemma4 lit '{r.gemma4 or 'vide'}'"
                        + (" comme le PMQ" if r.gemma4 == r.PMQ else " comme le PAD" if r.gemma4 == r.PAD else ""))
    if r.gemma4 and r.gemma4 != "ILLISIBLE":
        morceaux.append("absent des listes Insee" + (f" (proches : {r.candidats_insee})" if r.candidats_insee else "")
                        if r.absent_liste else "present dans les listes Insee")
    if r.consigne != r.gemma4:
        morceaux.append(f"la consigne relit '{r.consigne or 'vide'}' ({r.verdict_consigne.lower()})")
    if r.absent_apres_consigne:
        morceaux.append("envoye a un humain par l'usage 4")
    return " ; ".join(morceaux) + "."


D = T.copy()
D["UT"], D["numero_bulletin"] = D.cle.str.split("_").str[0], D.cle.str.split("_").str[1]
D = D.merge(APP[["cle", "chemin"]].rename(columns={"chemin": "image_page_S3"}), on="cle", how="left")
D["image_bloc"] = [str(BLOCS / f"{CAMPAGNE}_{k}.png") for k in D.cle]
D["verdict_gemma4"] = [verdict(v, t) for v, t in zip(D.gemma4, D.verite_arbitree)]
D["verdict_gemma4_Fiqual"] = [verdict(v, t) for v, t in zip(D.gemma4, D.PMQ)]
D["verdict_consigne"] = [verdict(v, t) for v, t in zip(D.consigne, D.verite_arbitree)]
D["verdict_PMQ"] = [verdict(v, t) for v, t in zip(D.PMQ, D.verite_arbitree)]
D["contradiction_gemma4"] = [contradiction(f, a) for f, a in zip(D["gemma4 seul | Fiqual"], D["gemma4 seul | arbitree"])]
D["contradiction_consigne"] = [contradiction(f, a) for f, a in
                               zip(D["3. consigne avec candidats | Fiqual"], D["3. consigne avec candidats | arbitree"])]
vrai_ou_pmq = [p if manquant(v) else v for p, v in zip(D.PMQ, D.verite_arbitree)]
D["type_erreur"] = [type_erreur(l, v) if l != v else "" for l, v in zip(D.gemma4, vrai_ou_pmq)]
D["lettres_fausses"] = [distance(l, v) if l not in ("", "ILLISIBLE") else None for l, v in zip(D.gemma4, vrai_ou_pmq)]
D["gemma4_comme"] = ["PMQ et PAD" if (g == p and g == a) else "PMQ" if g == p else "PAD" if g == a else "personne"
                     for g, p, a in zip(D.gemma4, D.PMQ, D.PAD)]
D["envoye_humain_signal"] = D.absent_liste.map({1: "oui", 0: "non"})
D["envoye_humain_usage4"] = D.absent_apres_consigne.map({1: "oui", 0: "non"})
D["verite_arbitree"] = D.verite_arbitree.map(lambda v: "(inconnue)" if manquant(v) else v)
D["commentaire"] = [commentaire(r) for r in D.itertuples()]
probleme = ((D.verdict_gemma4 != "JUSTE") | (D.verdict_consigne != "JUSTE") | (D.verdict_PMQ != "JUSTE")
            | (D.groupe != "A") | (D.contradiction_gemma4 != ""))
D["a_regarder"] = probleme.map({True: "oui", False: "non"})
ordre_v = {"TROMPE": 0, "VIDE (rien lu)": 1, "ILLISIBLE (avoue)": 2, "VERITE INCONNUE": 3, "JUSTE": 4}
D = D.sort_values(["a_regarder", "groupe", "verdict_gemma4", "cle", "champ"],
                  key=lambda c: c.map(ordre_v) if c.name == "verdict_gemma4" else c,
                  ascending=[False, True, True, True, True]).reset_index(drop=True)
D.insert(0, "n", range(1, len(D) + 1))

COLONNES_DETAIL = ["n", "a_regarder", "UT", "numero_bulletin", "champ", "groupe", "statut",
                   "PMQ", "PAD", "SeRN", "verite_arbitree", "lu", "gemma4", "verdict_gemma4", "verdict_gemma4_Fiqual",
                   "contradiction_gemma4", "type_erreur", "lettres_fausses", "gemma4_comme",
                   "absent_liste", "candidats_insee", "envoye_humain_signal", "consigne", "verdict_consigne",
                   "contradiction_consigne", "envoye_humain_usage4", "verdict_PMQ", "commentaire",
                   "cle", "image_page_S3", "image_bloc"]
DETAIL = D[COLONNES_DETAIL].rename(columns={"lu": "gemma4_brut", "PMQ": "PMQ (= Fiqual)",
                                            "absent_liste": "absent_liste_insee"})
A_REGARDER = DETAIL[DETAIL.a_regarder == "oui"].reset_index(drop=True)

LISEZ_MOI = pd.DataFrame([
    ("n", "numero de la ligne dans ce fichier"),
    ("a_regarder", "oui si gemma4, la consigne ou le PMQ se trompe, ou s'il y a une divergence PAD / PMQ"),
    ("UT, numero_bulletin", "identifient le bulletin (cle = UT_numero) ; image_page_S3 = la page complete sur S3"),
    ("champ", "NOM ou PRENOM"),
    ("groupe", "A = pas de divergence (PAD = PMQ) ; B = divergence arbitree par le SeRN ; C = divergence non arbitree"),
    ("PMQ (= Fiqual)", "ce qu'a saisi le PMQ : c'est la valeur du Fiqual"),
    ("PAD", "ce qu'a saisi le PAD (Ficod) ; egal au PMQ dans le groupe A"),
    ("SeRN", "la valeur retenue par le SeRN (groupe B seulement)"),
    ("verite_arbitree", "la reference retenue : PAD = PMQ (A), SeRN (B), inconnue (C)"),
    ("gemma4_brut", "la reponse exacte du modele, avant normalisation"),
    ("gemma4", "la reponse normalisee (majuscules, sans accent ni ponctuation) : c'est elle qui est comparee"),
    ("verdict_gemma4", "JUSTE / TROMPE (une autre valeur) / VIDE (rien lu) / ILLISIBLE (avoue) / VERITE INCONNUE, "
                       "par rapport a la verite arbitree"),
    ("verdict_gemma4_Fiqual", "le meme verdict, mais par rapport au Fiqual (ancienne mesure)"),
    ("contradiction_gemma4", "rempli quand les deux verdicts ne disent pas la meme chose : c'est la reponse a "
                             "« parmi les justes, y en a-t-il de faux ? »"),
    ("type_erreur", "1 lettre, 2 lettres, espace ou tiret, nom compose incomplet, illisible, autre"),
    ("lettres_fausses", "distance de Levenshtein : nombre minimal de lettres a changer, ajouter ou retirer"),
    ("gemma4_comme", "gemma4 a lu comme le PMQ, comme le PAD, les deux, ou personne"),
    ("absent_liste_insee", "1 si la lecture de gemma4 n'existe pas dans les listes de l'Insee"),
    ("candidats_insee", "les valeurs existantes les plus proches (montrees au modele dans la consigne)"),
    ("envoye_humain_signal", "oui si l'usage 1 (signal) envoie cette ligne a un humain"),
    ("consigne", "la lecture apres la relecture avec candidats (usage 3)"),
    ("envoye_humain_usage4", "oui si, apres la consigne, la valeur reste absente des listes : l'usage 4 l'envoie a un humain"),
    ("verdict_PMQ", "le PMQ lui-meme est-il juste par rapport a la verite arbitree ?"),
    ("commentaire", "la ligne racontee en une phrase"),
    ("image_bloc", "le fichier image du bloc Nom / Prenom (dossier /tmp/arbitrage/blocs du service)"),
], columns=["colonne", "signification"])

# --- les images : on fabrique le bloc de chaque bulletin a regarder (lecture S3, pas d'appel au modele)
MINI = Path("/tmp/arbitrage/miniatures")
MINI.mkdir(parents=True, exist_ok=True)
mini = {}
for cle in list(dict.fromkeys(A_REGARDER.cle))[:MAX_IMAGES]:
    try:
        im = bloc(cle).convert("RGB")
        r_ = LARGEUR_IMAGE / im.width
        im = im.resize((LARGEUR_IMAGE, max(1, int(im.height * r_))))
        fm = MINI / f"{CAMPAGNE}_{cle}.png"
        im.save(fm)
        mini[cle] = fm
    except Exception as e:
        print(f"image impossible pour {cle} : {type(e).__name__}")
print(f"{len(A_REGARDER)} lignes a regarder sur {len(DETAIL)} | {len(mini)} images de blocs")

# --- Excel
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Alignment, Font, PatternFill

f_detail = RESULTATS / f"arbitrage_{CAMPAGNE}_detail_lignes{SUFFIXE}.xlsx"
with pd.ExcelWriter(f_detail, engine="openpyxl") as w:
    LISEZ_MOI.to_excel(w, sheet_name="LISEZ_MOI", index=False)
    A_REGARDER.insert(1, "IMAGE", "")
    A_REGARDER.to_excel(w, sheet_name="a_regarder_avec_images", index=False)
    DETAIL.to_excel(w, sheet_name="toutes_les_lignes", index=False)
    DECOMPOSITION.to_excel(w, sheet_name="decomposition_groupes", index=False)

wb = load_workbook(f_detail)
COULEUR = {"JUSTE": "C6EFCE", "TROMPE": "FFC7CE", "VIDE (rien lu)": "FFEB9C",
           "ILLISIBLE (avoue)": "FFEB9C", "VERITE INCONNUE": "D9D9D9"}
for nom_f in ["a_regarder_avec_images", "toutes_les_lignes"]:
    ws = wb[nom_f]
    entetes = [c.value for c in ws[1]]
    for c in ws[1]:
        c.font = Font(bold=True)
        c.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "C2" if nom_f.startswith("a_") else "B2"
    ws.auto_filter.ref = ws.dimensions
    for j, e in enumerate(entetes, 1):
        lettre = ws.cell(1, j).column_letter
        ws.column_dimensions[lettre].width = {"IMAGE": LARGEUR_IMAGE / 7, "commentaire": 70, "image_page_S3": 40,
                                              "candidats_insee": 30, "contradiction_gemma4": 30,
                                              "contradiction_consigne": 30}.get(e, 14)
        if str(e).startswith("verdict"):
            for i in range(2, ws.max_row + 1):
                v = ws.cell(i, j).value
                if v in COULEUR:
                    ws.cell(i, j).fill = PatternFill("solid", fgColor=COULEUR[v])
    jc = entetes.index("commentaire") + 1
    for i in range(2, ws.max_row + 1):
        ws.cell(i, jc).alignment = Alignment(wrap_text=True, vertical="top")
    if nom_f.startswith("a_"):
        jk = entetes.index("cle") + 1
        for i in range(2, ws.max_row + 1):
            fm = mini.get(ws.cell(i, jk).value)
            if fm:
                img = XLImage(str(fm))
                img.anchor = f"B{i}"
                ws.add_image(img)
                ws.row_dimensions[i].height = img.height * 0.75 + 4
ws = wb["LISEZ_MOI"]
ws.column_dimensions["A"].width, ws.column_dimensions["B"].width = 26, 110
wb.save(f_detail)
print("Detail Excel :", f_detail)

# --- la meme chose en page HTML (images integrees : un seul fichier a telecharger)
import html as _html
def _img64(cle):
    fm = mini.get(cle)
    return (f'<img src="data:image/png;base64,{base64.b64encode(fm.read_bytes()).decode()}" width="{LARGEUR_IMAGE}">'
            if fm else "(pas d'image)")
lignes_html = []
for _, r in A_REGARDER.iterrows():
    cl = {"JUSTE": "ok", "TROMPE": "ko", "VERITE INCONNUE": "nsp"}.get(r.verdict_gemma4, "av")
    lignes_html.append(
        f"<tr><td>{r.n}</td><td>{_img64(r.cle)}<br><small>{_html.escape(r.cle)}</small></td>"
        f"<td><b>{r.champ}</b><br>groupe {r.groupe}</td>"
        f"<td>PMQ : <b>{_html.escape(r['PMQ (= Fiqual)'])}</b><br>PAD : {_html.escape(r.PAD)}<br>"
        f"SeRN : {_html.escape(r.SeRN) or '-'}<br>verite : <b>{_html.escape(r.verite_arbitree)}</b></td>"
        f"<td>gemma4 : <b>{_html.escape(r.gemma4) or 'vide'}</b><br>consigne : {_html.escape(r.consigne) or 'vide'}</td>"
        f"<td class='{cl}'>{_html.escape(r.verdict_gemma4)}<br><small>{_html.escape(r.contradiction_gemma4)}</small></td>"
        f"<td>{_html.escape(r.commentaire)}</td></tr>")
page = f"""<!doctype html><html lang="fr"><head><meta charset="utf-8"><title>Lignes a regarder {CAMPAGNE}</title>
<style>body{{font-family:sans-serif;margin:16px}}table{{border-collapse:collapse}}td,th{{border:1px solid #ccc;
padding:6px;vertical-align:top;font-size:14px}}th{{background:#eee;position:sticky;top:0}}.ok{{background:#c6efce}}
.ko{{background:#ffc7ce}}.av{{background:#ffeb9c}}.nsp{{background:#ddd}}</style></head><body>
<h1>{CAMPAGNE} : {len(A_REGARDER)} lignes a regarder (sur {len(DETAIL)})</h1>
<p>Groupe A = pas de divergence PAD / PMQ ; B = divergence arbitree par le SeRN ; C = divergence non arbitree.</p>
<table><tr><th>n</th><th>bloc</th><th>champ</th><th>saisies humaines</th><th>lectures</th><th>verdict gemma4</th>
<th>commentaire</th></tr>{''.join(lignes_html)}</table></body></html>"""
f_html = RESULTATS / f"arbitrage_{CAMPAGNE}_galerie{SUFFIXE}.html"
f_html.write_text(page, encoding="utf-8")
print("Galerie HTML :", f_html, "\n(telecharger, puis ouvrir dans Firefox)")
with pd.option_context("display.max_rows", 30, "display.max_colwidth", 120):
    display(A_REGARDER[["n", "champ", "groupe", "verdict_gemma4", "contradiction_gemma4", "commentaire"]].head(30))

59 lignes a regarder sur 335 | 51 images de blocs
Detail Excel : /home/onyxia/work/banc-htr/resultats/arbitrage_RG25_detail_lignes.xlsx
Galerie HTML : /home/onyxia/work/banc-htr/resultats/arbitrage_RG25_galerie.html 
(telecharger, puis ouvrir dans Firefox)


,n,champ,groupe,verdict_gemma4,contradiction_gemma4,commentaire
0,1,PRENOM,A,TROMPE,,gemma4 lit 'EVELYNE' au lieu de 'PARO EVELYNE' (nom compose incomplet ou en trop) ; present dans les listes Insee.
1,2,PRENOM,A,TROMPE,,"gemma4 lit 'ANTOME' au lieu de 'ANTOINE' (2 lettres) ; absent des listes Insee (proches : ANTOINE, ANTONY, ANATOLE, ..."
2,3,NOM,A,TROMPE,,gemma4 lit 'JANET' au lieu de 'JAMET' (1 lettre) ; present dans les listes Insee.
3,4,PRENOM,A,TROMPE,,gemma4 lit 'CELINE' au lieu de 'CEDRIC' (autre (3 lettres ou plus)) ; present dans les listes Insee.
4,5,NOM,A,TROMPE,,"gemma4 lit 'GTAGNOL' au lieu de 'STAGNOL' (1 lettre) ; absent des listes Insee (proches : GATIGNOL, GAGNON, BAGNOL, ..."
5,6,PRENOM,A,TROMPE,,"gemma4 lit 'M LENA' au lieu de 'MILENA' (1 lettre) ; absent des listes Insee (proches : LENA, ELENA, MELINA, MYLENE,..."
6,7,PRENOM,A,TROMPE,,gemma4 lit 'CELINE' au lieu de 'CEDRIC' (autre (3 lettres ou plus)) ; present dans les listes Insee.
7,8,NOM,A,TROMPE,,gemma4 lit 'VIC' au lieu de 'VIL' (1 lettre) ; present dans les listes Insee.
8,9,NOM,A,TROMPE,,"gemma4 lit 'BESONNE' au lieu de 'BESSONNE' (1 lettre) ; absent des listes Insee (proches : BONNE, PERONNE, PERSONNE,..."
9,10,PRENOM,A,TROMPE,,"gemma4 lit 'PIENE' au lieu de 'PIERRE' (2 lettres) ; absent des listes Insee (proches : PIERRE, RENE, IRENE, REINE, ..."


In [39]:
# 13 bis. VERIFIER LES DEUX MLFLOW avant d'enregistrer (diagnostic, ne modifie rien)
import importlib
sys.path.insert(0, str(RACINE / "src"))
try:
    from mesure_qualite.evaluation import double_mlflow as dm
    dm = importlib.reload(dm)                       # toujours la version du disque, pas une ancienne en memoire
    print("fichier utilise      :", dm.__file__)
    print("version a jour       :", "oui" if hasattr(dm, "_lire_vault_http") else
          "NON : redeposer double_mlflow.py dans src/mesure_qualite/evaluation/ (Overwrite)")
    print("commande vault       :", shutil.which("vault") or "absente (normal : lecture par l'API web)")
    print("VAULT_ADDR           :", os.environ.get("VAULT_ADDR"))
    print("VAULT_TOKEN present  :", "VAULT_TOKEN" in os.environ)
    print("MLflow personnel     :", os.environ.get("MLFLOW_TRACKING_URI"))
    CIBLES_MLFLOW = list(dm.cibles().keys())
    print("cibles               :", CIBLES_MLFLOW, "(attendu : ['perso', 'partage'])")
except ImportError as e:
    print("module double_mlflow introuvable :", e)

fichier utilise      : /home/onyxia/work/mesure-qualite-rp/src/mesure_qualite/evaluation/double_mlflow.py
version a jour       : oui
commande vault       : absente (normal : lecture par l'API web)
VAULT_ADDR           : https://vault.lab.sspcloud.fr
VAULT_TOKEN present  : True
MLflow personnel     : https://user-farisazibert-mlflow.user.lab.sspcloud.fr
cibles               : ['perso', 'partage'] (attendu : ['perso', 'partage'])


In [40]:
# 14. Fichier Excel d'analyse, et MLflow (personnel + partage)
f_xlsx = RESULTATS / f"arbitrage_{CAMPAGNE}_analyse{SUFFIXE}.xlsx"
with pd.ExcelWriter(f_xlsx) as w:
    SYNTHESE.to_excel(w, sheet_name="1_synthese", index=False)
    DECOMPOSITION.to_excel(w, sheet_name="1b_decomposition", index=False)
    BARRE.to_excel(w, sheet_name="2_barre_PMQ", index=False)
    DETAIL_DIV.to_excel(w, sheet_name="3_divergences", index=False)
    TYPES.reset_index().to_excel(w, sheet_name="4_types_erreurs", index=False)
    ERREURS.to_excel(w, sheet_name="5_erreurs_gemma4", index=False)
    T.to_excel(w, sheet_name="6_toutes_les_lignes", index=False)
SYNTHESE.to_csv(RESULTATS / f"arbitrage_{CAMPAGNE}_resultat{SUFFIXE}.csv", index=False)
print("Excel :", f_xlsx, "\n(explorateur JupyterLab : banc-htr/resultats -> clic droit -> Download)")


def enregistrer():
    with mlflow.start_run(run_name=f"arbitrage-sern-{CAMPAGNE}") as run:
        mlflow.set_tags({"etude": "comparaison-pmq", "notebook": "10", "statut": "exploratoire",
                         "campagne": CAMPAGNE, "modele": MODELE})
        mlflow.log_params({"lignes": len(T), "divergences": len(DV), "non_arbitrees": n_non_arb,
                           "variable_nom": VARIABLE_NOM, "variable_prenom": VARIABLE_PRENOM})
        mlflow.log_metrics({"pmq_erreur_min": bas, "pmq_erreur_max": haut})
        for _, r in SYNTHESE.iterrows():
            nom = re.sub(r"[^a-z0-9]+", "_", f"{r['usage']}_{r['verite']}".lower()).strip("_")
            mlflow.log_metrics({f"{nom}_justes_pct": float(r["justes_%"]),
                                f"{nom}_silencieuses_pct": float(r["silencieuses_%"]),
                                f"{nom}_humain_pct": float(r["a_un_humain_%"])})
        for f_ in [f_xlsx, f_detail, f_html]:
            mlflow.log_artifact(str(f_))
    return run.info.run_id[:8]


if TEST_RAPIDE:
    print("Mode TEST RAPIDE : rien n'est enregistre dans MLflow.")
else:
    try:
        sys.path.insert(0, str(RACINE / "src"))
        from mesure_qualite.evaluation import double_mlflow as dm
        print(dm.partout(enregistrer))
    except ImportError:
        mlflow.set_experiment("fiqual-banc-essai")
        print("MLflow personnel seulement :", enregistrer())

Excel : /home/onyxia/work/banc-htr/resultats/arbitrage_RG25_analyse.xlsx 
(explorateur JupyterLab : banc-htr/resultats -> clic droit -> Download)
🏃 View run arbitrage-sern-RG25 at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/31d0c9d67d4942ce91f25a055987591d
🧪 View experiment at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/1
MLflow perso    : enregistre (https://user-farisazibert-mlflow.user.lab.sspcloud.fr)
🏃 View run arbitrage-sern-RG25 at: https://projet-mesure-qualite-rp-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/59dbdaa8f5b249ba90143dca58fbae1b
🧪 View experiment at: https://projet-mesure-qualite-rp-mlflow.user.lab.sspcloud.fr/#/experiments/1
MLflow partage  : enregistre (https://projet-mesure-qualite-rp-mlflow.user.lab.sspcloud.fr/)
{'perso': '31d0c9d6', 'partage': '59dbdaa8'}


In [41]:
# 15. PHRASES PRETES A DIRE (calculees a partir des chiffres ci-dessus)
def ligne_s(usage, verite):
    return SYNTHESE[(SYNTHESE.usage == usage) & (SYNTHESE.verite == verite)].iloc[0]
def ligne_d(usage):
    return DECOMPOSITION[DECOMPOSITION.usage == usage].iloc[0]
g_f, g_a = ligne_s("gemma4 seul", "Fiqual"), ligne_s("gemma4 seul", "arbitree")
c_a, s_a = ligne_s("3. consigne avec candidats", "arbitree"), ligne_s("1. signal (relecture des absents)", "arbitree")
q_a = ligne_s("4. consigne + signal", "arbitree")
dg = ligne_d("gemma4 seul").tolist()
nA, nB, nC = (int((T.groupe == x).sum()) for x in "ABC")
print(f"1. Sur {len(T)} lignes, {nA} n'ont aucune divergence entre le PAD et le PMQ : la, juger par rapport au "
      f"Fiqual ou par rapport a l'arbitrage donne exactement le meme resultat.")
print(f"2. Les differences viennent des {nB + nC} divergences : {nB} arbitrees par le SeRN, {nC} non arbitrees, "
      f"donc ecartees.")
print(f"3. gemma4 seul : {g_f['justes_%']} % de justes par rapport au Fiqual, {g_a['justes_%']} % par rapport a la "
      f"verite arbitree. Parmi ses {dg[1]} justes Fiqual, {dg[4]} sont en fait fausses (il a recopie l'erreur du PMQ) "
      f"et {dg[5]} tombent sur des cas non arbitres ; inversement, {dg[6]} lignes comptees fausses etaient justes.")
print(f"4. Le PMQ fait entre {bas:.1%} et {haut:.1%} d'erreurs ; gemma4 seul fait {g_a['silencieuses_%']} % "
      f"d'erreurs silencieuses.")
print(f"5. La consigne avec candidats descend a {c_a['silencieuses_%']} % ; le signal seul a {s_a['silencieuses_%']} % "
      f"(en envoyant {s_a['a_un_humain_%']} % des lignes a un humain).")
print(f"6. Consigne + signal : {q_a['silencieuses_%']} % d'erreurs silencieuses, en envoyant {q_a['a_un_humain_%']} % "
      f"des lignes a un humain. Position par rapport au PMQ : "
      f"{BARRE.loc[BARRE.usage == '4. consigne + signal', 'position'].iloc[0]}.")
print(f"7. Le detail ligne par ligne, avec l'image de chaque bloc, est dans {f_detail.name} et {f_html.name}.")

1. Sur 335 lignes, 313 n'ont aucune divergence entre le PAD et le PMQ : la, juger par rapport au Fiqual ou par rapport a l'arbitrage donne exactement le meme resultat.
2. Les differences viennent des 22 divergences : 5 arbitrees par le SeRN, 17 non arbitrees, donc ecartees.
3. gemma4 seul : 83.9 % de justes par rapport au Fiqual, 88.1 % par rapport a la verite arbitree. Parmi ses 281 justes Fiqual, 0 sont en fait fausses (il a recopie l'erreur du PMQ) et 4 tombent sur des cas non arbitres ; inversement, 3 lignes comptees fausses etaient justes.
4. Le PMQ fait entre 1.6% et 6.6% d'erreurs ; gemma4 seul fait 11.9 % d'erreurs silencieuses.
5. La consigne avec candidats descend a 9.4 % ; le signal seul a 6.0 % (en envoyant 12.6 % des lignes a un humain).
6. Consigne + signal : 6.9 % d'erreurs silencieuses, en envoyant 9.1 % des lignes a un humain. Position par rapport au PMQ : au-dessus du PMQ.
7. Le detail ligne par ligne, avec l'image de chaque bloc, est dans arbitrage_RG25_detail_lignes